## 0 · Setup & Ambiente

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys

REPO_PATH = '/content/classificador-fake-br'
if not os.path.exists(REPO_PATH):
    os.system(f'git clone https://github.com/lisearantes/portuguese-fake-news-detection.git {REPO_PATH}')
else:
    os.system(f'git -C {REPO_PATH} pull --quiet')

sys.path.insert(0, REPO_PATH)

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', '-r', f'{REPO_PATH}/requirements.txt'])
os.system('python -m spacy download pt_core_news_sm')

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

print('Dependências instaladas.')

In [ ]:
import matplotlib.font_manager as fm
import pandas as pd

os.system('wget -q -O Arial.ttf https://github.com/matomo-org/travis-scripts/raw/master/fonts/Arial.ttf')
fm.fontManager.addfont('Arial.ttf')

from src.visualization import configurar_fonte_arial
configurar_fonte_arial()

pd.set_option('display.max_colwidth', 140)
print('Setup concluído.')

## 1 · Carregamento do Corpus

In [ ]:
CSV_PATH  = '/content/drive/MyDrive/Fake.br-Corpus/fake.br-full_texts.csv'
FULL_PATH = '/content/drive/MyDrive/Fake.br-Corpus/full_texts/'

if not os.path.exists(FULL_PATH):
    raise FileNotFoundError(f'Corpus não encontrado em: {FULL_PATH}')

df = pd.read_csv(CSV_PATH)
print(f'Dimensões: {df.shape}')
print(f'Colunas: {list(df.columns)}')
print(f'Tipos:\n{df.dtypes}')

## 2 · Análise Exploratória (EDA)

In [ ]:
from src.visualization import (
    plot_preenchimento,
    plot_distribuicao_classes,
    plot_distribuicao_categorias,
)

plot_preenchimento(df)
plot_distribuicao_classes(df)
plot_distribuicao_categorias(df)

## 3 · Pré-processamento de Texto

In [ ]:
from src.preprocessing import text_cleaning, wrap_texto

df['clean_text'] = df['texto_completo'].apply(text_cleaning)

In [ ]:
texto_original = df['texto_completo'].dropna().astype(str).iloc[0]
texto_limpo    = text_cleaning(texto_original)
n_orig  = len(texto_original.split())
n_limpo = len(texto_limpo.split())
compactacao = (n_orig - n_limpo) / n_orig * 100

print(f'Original ({n_orig} palavras):\n{wrap_texto(texto_original)}\n')
print(f'Limpo ({n_limpo} palavras | compactação: {compactacao:.1f}%):\n{wrap_texto(texto_limpo)}')

In [ ]:
from src.visualization import plot_boxplot_tamanho
plot_boxplot_tamanho(df)

## 4 · Vetorização TF-IDF

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

X = df['clean_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

vectorizer = TfidfVectorizer(
    lowercase=True,
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf  = vectorizer.transform(X_test)
tfidf_terms   = vectorizer.get_feature_names_out()

print(f'Treino:    {X_train_tfidf.shape[0]} docs x {X_train_tfidf.shape[1]} termos')
print(f'Teste:     {X_test_tfidf.shape[0]} docs x {X_test_tfidf.shape[1]} termos')
print(f'Densidade: {X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1]):.6f}')

In [ ]:
from src.visualization import plot_esparsidade_tfidf, plot_heatmap_tfidf

plot_esparsidade_tfidf(X_train_tfidf)
plot_heatmap_tfidf(X_train_tfidf, tfidf_terms)

## 5 · Modelagem e Avaliação

In [ ]:
from src.evaluation import treinar_svc, treinar_lr, relatorio_completo

modelo_svc, pred_svc = treinar_svc(X_train_tfidf, y_train, X_test_tfidf)
modelo_lr,  pred_lr  = treinar_lr(X_train_tfidf,  y_train, X_test_tfidf)

relatorio_completo(y_test, pred_svc, 'Linear SVC')
relatorio_completo(y_test, pred_lr,  'Logistic Regression')

In [ ]:
from src.evaluation import tabela_resultados

df_resultados = tabela_resultados(
    y_test,
    [(pred_svc, 'Linear SVC'), (pred_lr, 'Logistic Regression')],
)
print(df_resultados.to_string(index=False))

In [ ]:
from src.visualization import plot_confusion_matrix

plot_confusion_matrix(y_test, pred_svc)